In [1]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import load_npz

matrix = load_npz("../data/processed/interaction_matrix.npz")
interactions = pd.read_parquet("../data/processed/interactions.parquet")

with open("../data/processed/user_encoder.pkl", "rb") as f:
    user_enc = pickle.load(f)

with open("../data/processed/artist_encoder.pkl", "rb") as f:
    artist_enc = pickle.load(f)

print(f"Matrix: {matrix.shape}, nnz: {matrix.nnz:,}")

Matrix: (74999, 145778), nnz: 3,664,980


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommenders.popularity import PopularityRecommender
from src.recommenders.collabarative import CollaborativeFilteringRecommender

cf_rec = CollaborativeFilteringRecommender(n_neighbors=50)
cf_rec.fit(matrix, user_enc, artist_enc, interactions)

CF Recommender fitted.
Matrix: 74,999 users x 145,778 artists
Active artists (>=5 listeners): 11,660
Reverse artist lookup precomputed: 145,778 entries


In [3]:
# Use the same user you inspected in Phase 5 for direct comparison
sample_user = interactions["user_id"].iloc[100]

print(f"User: {sample_user[:16]}...")
print(f"\nThis user listens to:")
print(
    interactions[interactions["user_id"] == sample_user]
    .sort_values("plays", ascending=False)
    [["artist_name", "plays"]]
    .head(8)
    .to_string(index=False)
)

User: 0000f687d4fe9c1e...

This user listens to:
    artist_name  plays
black eyed peas     12
  nelly furtado     11
 antonio orozco     10
          decai     10
        rihanna     10
   kiko & shara      9
   marc anthony      8
           mika      7


In [4]:
import time 
start = time.time()
user_idx = user_enc.transform([sample_user])[0]
user_profile = matrix[user_idx]

scores = np.asarray(matrix.sum(axis=0)).ravel().astype(float)
seen_items = user_profile.indices
scores[seen_items] = -np.inf

top_idx = np.argpartition(scores, -10)[-10:]
top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

ib_recs = pd.DataFrame(
    {
        "artist_name": artist_enc.inverse_transform(top_idx),
        "score": scores[top_idx],
    }
)
elapsed = time.time() - start

print(f"Item-Based CF Recommendations ({elapsed:.2f}s):")
print(ib_recs.to_string(index=False))

Item-Based CF Recommendations (0.11s):
          artist_name       score
          the beatles 2439.426025
            radiohead 2426.516113
             coldplay 2058.456299
red hot chili peppers 1484.701538
                 muse 1447.516235
            metallica 1413.650879
           pink floyd 1376.829712
          linkin park 1243.401733
          the killers 1240.323975
              nirvana 1188.629517


In [5]:
from src.recommenders.collabarative import CollaborativeFilteringRecommender
from scipy.sparse import load_npz
import pickle
import pandas as pd

matrix = load_npz("../data/processed/interaction_matrix.npz")
interactions = pd.read_parquet("../data/processed/interactions.parquet")

with open("../data/processed/user_encoder.pkl", "rb") as f:
    user_enc = pickle.load(f)
with open("../data/processed/artist_encoder.pkl", "rb") as f:
    artist_enc = pickle.load(f)

cf_rec = CollaborativeFilteringRecommender(n_neighbors=50)
cf_rec.fit(matrix, user_enc, artist_enc, interactions)

with open("../data/processed/cf_recommender.pkl", "wb") as f:
    pickle.dump(cf_rec, f)

CF Recommender fitted.
Matrix: 74,999 users x 145,778 artists
Active artists (>=5 listeners): 11,660
Reverse artist lookup precomputed: 145,778 entries


In [6]:
import time
import numpy as np

start = time.time()
user_vec = cf_rec._get_user_vector(sample_user)
user_idx = cf_rec.user_enc.transform([sample_user])[0]
print(f"Step 1 - get user vector:     {time.time()-start:.3f}s")

start = time.time()
sim_scores = np.asarray(cf_rec.matrix.dot(user_vec.T).todense()).flatten()
print(f"Step 2 - sparse dot product:  {time.time()-start:.3f}s")

start = time.time()
sim_scores[user_idx] = -1
neighbor_indices = np.argsort(sim_scores)[::-1][:50]
print(f"Step 3 - argsort neighbors:   {time.time()-start:.3f}s")

start = time.time()
heard = set(
    cf_rec.interactions_df.loc[
        cf_rec.interactions_df["user_id"] == sample_user, "artist_name"
    ].values
)
neighbor_sims = sim_scores[neighbor_indices]
candidate_scores = {}

for neighbor_idx, sim in zip(neighbor_indices, neighbor_sims):
    if sim <= 0:
        continue
    neighbor_row = cf_rec.matrix.getrow(neighbor_idx)
    artist_indices = neighbor_row.indices
    artist_values = neighbor_row.data
    for art_idx, art_val in zip(artist_indices, artist_values):
        artist_name = artist_enc.inverse_transform([art_idx])[0]
        if artist_name in heard:
            continue
        if artist_name not in candidate_scores:
            candidate_scores[artist_name] = 0.0
        candidate_scores[artist_name] += float(sim) * float(art_val)

print(f"Step 4 - candidate loop:      {time.time()-start:.3f}s")
print(f"Candidates found: {len(candidate_scores)}")

Step 1 - get user vector:     0.087s
Step 2 - sparse dot product:  0.014s
Step 3 - argsort neighbors:   0.006s


KeyboardInterrupt: 

In [ ]:
import time
import numpy as np

# Rebuild heard set
heard = set(
    cf_rec.interactions_df.loc[
        cf_rec.interactions_df["user_id"] == sample_user, "artist_name"
    ].values
)

# Get neighbors
user_vec = cf_rec._get_user_vector(sample_user)
user_idx = cf_rec.user_enc.transform([sample_user])[0]
sim_scores = np.asarray(cf_rec.matrix.dot(user_vec.T).todense()).flatten()
sim_scores[user_idx] = -1
neighbor_indices = np.argsort(sim_scores)[::-1][:50]
neighbor_sims = sim_scores[neighbor_indices]

# Time individual operations on first neighbor only
neighbor_idx = neighbor_indices[0]
sim = neighbor_sims[0]

start = time.time()
for _ in range(1000):
    neighbor_row = cf_rec.matrix.getrow(neighbor_idx)
print(f"getrow x1000:           {time.time()-start:.3f}s")

start = time.time()
for _ in range(1000):
    artist_indices = neighbor_row.indices
    artist_values = neighbor_row.data
print(f".indices/.data x1000:  {time.time()-start:.3f}s")

start = time.time()
for _ in range(1000):
    artist_name = artist_enc.inverse_transform([42])[0]
print(f"idx_to_artist x1000:   {time.time()-start:.3f}s")

start = time.time()
for _ in range(1000):
    _ = 42 in heard
print(f"set lookup x1000:      {time.time()-start:.3f}s")

# How many artists per neighbor on average?
total_artists = 0
for idx in neighbor_indices:
    total_artists += cf_rec.matrix.getrow(idx).nnz
print(f"\nAvg artists per neighbor: {total_artists/len(neighbor_indices):.0f}")
print(f"Total iterations in loop: {total_artists:,}")

getrow x1000:           0.007s
.indices/.data x1000:  0.000s
idx_to_artist x1000:   29.304s
set lookup x1000:      0.000s

Avg artists per neighbor: 50
Total iterations in loop: 2,505


In [ ]:
print(hasattr(cf_rec, 'matrix_T_filtered'))
print(hasattr(cf_rec, 'active_artist_indices'))

In [ ]:
import pickle
import time

# Load CB recommender
with open("../data/processed/cb_recommender.pkl", "rb") as f:
    cb_rec = pickle.load(f)

# Run all three engines on the same user
cb_recs = cb_rec.recommend_for_user(sample_user, k=10)

start = time.time()
ub_recs = cf_rec.recommend_user_based(sample_user, k=10)
ub_time = time.time() - start

start = time.time()
ib_recs = cf_rec.recommend_item_based(sample_user, k=10)
ib_time = time.time() - start

print(f"User-Based CF query time:  {ub_time:.2f}s")
print(f"Item-Based CF query time:  {ib_time:.2f}s")

# Three-way comparison table
print("\n" + "=" * 65)
print(f"{'CONTENT-BASED':<22} {'USER-BASED CF':<22} {'ITEM-BASED CF':<22}")
print("=" * 65)
for i in range(10):
    cb_art  = cb_recs.iloc[i]["artist_name"]  if i < len(cb_recs)  else "-"
    ub_art  = ub_recs.iloc[i]["artist_name"]  if i < len(ub_recs)  else "-"
    ib_art  = ib_recs.iloc[i]["artist_name"]  if i < len(ib_recs)  else "-"
    print(f"{cb_art:<22} {ub_art:<22} {ib_art:<22}")

# Pairwise overlaps
cb_set = set(cb_recs["artist_name"])
ub_set = set(ub_recs["artist_name"])
ib_set = set(ib_recs["artist_name"])

print(f"\nCB ∩ User-CF overlap:  {len(cb_set & ub_set)}/10")
print(f"CB ∩ Item-CF overlap:  {len(cb_set & ib_set)}/10")
print(f"User-CF ∩ Item-CF:     {len(ub_set & ib_set)}/10")

KeyboardInterrupt: 

In [ ]:
import inspect

print(inspect.getsource(cf_rec.recommend_user_based))

    def recommend_user_based(
        self,
        user_id: str,
        k: int = 10,
        filter_seen: bool = True
    ) -> pd.DataFrame:

        assert self.is_fitted, "Call fit() first"

        user_vec = self._get_user_vector(user_id)

        if user_vec is None:
            print(f"User '{user_id[:16]}...' not found in interaction matrix.")
            return pd.DataFrame(
                columns=["artist_name", "cf_score", "recommendation_source"]
            )

        user_idx = self.user_enc.transform([user_id])[0]

        # Proper cosine similarity between target user and all users
        sim_scores = cosine_similarity(
            self.matrix,
            user_vec
        ).flatten()

        # Remove self from neighbors
        sim_scores[user_idx] = -1

        # Top similar users
        neighbor_indices = np.argsort(sim_scores)[::-1][:self.n_neighbors]
        neighbor_sims = sim_scores[neighbor_indices]

        # Artists already listened to by target user
    

In [ ]:
print(cf_rec.matrix.shape)

(74999, 145778)


In [ ]:
print(hasattr(cf_rec, 'matrix_T_filtered'))
print(hasattr(cf_rec, 'active_artist_indices'))

True
True


In [8]:
import pickle
import time

with open("../data/processed/cb_recommender.pkl", "rb") as f:
    cb_rec = pickle.load(f)

cb_recs = cb_rec.recommend_for_user(sample_user, k=10)
ub_recs = cf_rec.recommend_user_based(sample_user, k=10)
scores[seen_items] = -np.inf

print("=" * 65)
print(f"{'CONTENT-BASED':<22} {'USER-BASED CF':<22} {'ITEM-BASED CF':<22}")
print("=" * 65)
for i in range(10):
    cb_art = cb_recs.iloc[i]["artist_name"] if i < len(cb_recs) else "-"
    ub_art = ub_recs.iloc[i]["artist_name"] if i < len(ub_recs) else "-"
    ib_art = ib_recs.iloc[i]["artist_name"] if i < len(ib_recs) else "-"
    print(f"{cb_art:<22} {ub_art:<22} {ib_art:<22}")

cb_set = set(cb_recs["artist_name"])
ub_set = set(ub_recs["artist_name"])
ib_set = set(ib_recs["artist_name"])

print(f"\nCB ∩ User-CF overlap:  {len(cb_set & ub_set)}/10")
print(f"CB ∩ Item-CF overlap:  {len(cb_set & ib_set)}/10")
print(f"User-CF ∩ Item-CF:     {len(ub_set & ib_set)}/10")

CONTENT-BASED          USER-BASED CF          ITEM-BASED CF         
ponto de equilíbrio    britney spears         the beatles           
laurent wolf           el canto del loco      radiohead             
fantasy                shakira                coldplay              
miranda!               nena daconte           red hot chili peppers 
miranda                chenoa                 muse                  
mateus e cristiano     estopa                 metallica             
calvin harris          the pussycat dolls     pink floyd            
seventeen              conchita               linkin park           
dennis                 kylie minogue          the killers           
destra                 pereza                 nirvana               

CB ∩ User-CF overlap:  0/10
CB ∩ Item-CF overlap:  0/10
User-CF ∩ Item-CF:     0/10
